## 1. Data Loading and Preparation

In [ ]:
from pathlib import Path
import os
import pandas as pd
import time
import numpy as np

# Configuration
hospital_name = "hospital_1"
data_dir = Path("../data")
hospital_path = data_dir / hospital_name

In [ ]:
def load_omop_table(base_path, table_name, columns=None):
    """
    Load an OMOP table from a single Parquet file
    or from a directory containing multiple Parquet chunks.
    """
    full_path = Path(base_path) / table_name

    # Single Parquet file
    if full_path.is_file():
        return pd.read_parquet(full_path, columns=columns)

    # Directory containing multiple Parquet chunks
    all_chunks = []

    files = [
        f for f in os.listdir(full_path)
        if os.path.isfile(full_path / f)
    ]

    print(f"Loading {len(files)} chunks from {table_name}...")

    for f in files:
        try:
            chunk = pd.read_parquet(
                full_path / f,
                engine="pyarrow",
                columns=columns
            )

            # Normalize string-like columns before concatenation
            for col in chunk.columns:
                if chunk[col].dtype == "object" or "null" in str(chunk[col].dtype):
                    chunk[col] = (
                        chunk[col]
                        .astype(str)
                        .replace("None", pd.NA)
                        .replace("nan", pd.NA)
                    )

            all_chunks.append(chunk)

        except Exception as e:
            print(f"Skipping chunk {f}: {e}")

    if not all_chunks:
        print(f"No data found in {full_path}")
        return None

    return pd.concat(all_chunks, axis=0, ignore_index=True, sort=False)

In [ ]:
start_time = time.time()

# Load required OMOP tables
person_df = load_omop_table(hospital_path, "person")
visit_df = load_omop_table(hospital_path, "visit_occurrence")
visit_detail_df = load_omop_table(hospital_path, "visit_detail")
death_df = load_omop_table(hospital_path, "death")
condition_df = load_omop_table(hospital_path, "condition_occurrence")
observation_df = load_omop_table(hospital_path, "observation")
procedure_df = load_omop_table(hospital_path, "procedure_occurrence")
drug_df = load_omop_table(hospital_path, "drug_exposure")

# Load only the measurement columns required for feature extraction
measurement_df = load_omop_table(
    hospital_path,
    "measurement",
    columns=[
        "person_id",
        "visit_occurrence_id",
        "measurement_concept_name",
        "measurement_datetime",
        "measurement_date",
        "value_as_number",
    ],
)

end_time = time.time()

print(f"Time: {(end_time - start_time) / 60:.2f} minutes")
print("Finished loading tables successfully")

In [ ]:
# Convert date columns for temporal analysis
visit_df["visit_start_datetime"] = pd.to_datetime(
    visit_df["visit_start_datetime"], errors="coerce"
)

visit_df["visit_end_datetime"] = pd.to_datetime(
    visit_df["visit_end_datetime"], errors="coerce"
)

if "measurement_datetime" in measurement_df.columns:
    measurement_df["measurement_datetime"] = pd.to_datetime(
        measurement_df["measurement_datetime"], errors="coerce"
    )

if "measurement_date" in measurement_df.columns:
    measurement_df["measurement_date"] = pd.to_datetime(
        measurement_df["measurement_date"], errors="coerce"
    )

In [ ]:
# Standardize patient IDs across OMOP tables for consistent joins
person_df["person_id"] = person_df["person_id"].astype(str)
visit_df["person_id"] = visit_df["person_id"].astype(str)
condition_df["person_id"] = condition_df["person_id"].astype(str)
observation_df["person_id"] = observation_df["person_id"].astype(str)
procedure_df["person_id"] = procedure_df["person_id"].astype(str)
drug_df["person_id"] = drug_df["person_id"].astype(str)
measurement_df["person_id"] = measurement_df["person_id"].astype(str)

In [ ]:
start_time = time.time()

# Group OMOP records by patient for efficient feature extraction
persons_by_patient = dict(tuple(person_df.groupby("person_id")))
visits_by_patient = dict(tuple(visit_df.groupby("person_id")))
conditions_by_patient = dict(tuple(condition_df.groupby("person_id")))
observations_by_patient = dict(tuple(observation_df.groupby("person_id")))
procedures_by_patient = dict(tuple(procedure_df.groupby("person_id")))
drugs_by_patient = dict(tuple(drug_df.groupby("person_id")))
measurements_by_patient = dict(tuple(measurement_df.groupby("person_id")))

birth_year_by_patient = dict(
    zip(person_df["person_id"], person_df["year_of_birth"])
)

# Create visit-level lookup dictionaries
visit_type_map = dict(
    visit_df.set_index("visit_occurrence_id")["visit_concept_name"]
)

department_map = (
    visit_detail_df
    .groupby("visit_occurrence_id")["care_site_name"]
    .apply(lambda x: sorted(set(x.dropna().astype(str))))
    .to_dict()
)

end_time = time.time()

print(f"Time: {(end_time - start_time) / 60:.2f} minutes")
print("Finished creating dictionaries successfully")

In [ ]:
# Empty templates used when a patient has no records in a specific OMOP table
empty_persons = pd.DataFrame(columns=person_df.columns)
empty_visits = pd.DataFrame(columns=visit_df.columns)
empty_conditions = pd.DataFrame(columns=condition_df.columns)
empty_observations = pd.DataFrame(columns=observation_df.columns)
empty_procedures = pd.DataFrame(columns=procedure_df.columns)
empty_drugs = pd.DataFrame(columns=drug_df.columns)
empty_measurements = pd.DataFrame(columns=measurement_df.columns)

## 2. Hospitalization Construction


In [ ]:
def build_merged_hospitalizations(
    patient_visits,
    person_id,
    birth_year=None,
    visit_type_map=None,
    department_map=None,
    department_keywords=None
):
    """
    Build merged hospitalization episodes for a single patient.

    Visits that overlap or occur within 48 hours are treated as part
    of the same hospitalization episode.
    """

    output_columns = [
        "person_id",
        "merged_hospitalization_id",
        "hospitalization_number",
        "start_datetime",
        "end_datetime",
        "length_of_stay_days",
        "age_at_hospitalization",
        "visit_ids",
        "visit_types",
        "departments",
        "has_inpatient_visit",
        "has_relevant_department",
        "has_relevant_inpatient_visit"
    ]

    # Prepare visits in chronological order
    visits = patient_visits.copy()

    if visits.empty:
        return pd.DataFrame(columns=output_columns)

    visits = visits.dropna(
        subset=["visit_start_datetime", "visit_end_datetime"]
    )
    visits = visits.sort_values(
        "visit_start_datetime"
    ).reset_index(drop=True)

    if visits.empty:
        return pd.DataFrame(columns=output_columns)

    # Merge overlapping visits or visits separated by up to 48 hours
    merged_rows = []

    current_start = visits.loc[0, "visit_start_datetime"]
    current_end = visits.loc[0, "visit_end_datetime"]
    current_visit_ids = [visits.loc[0, "visit_occurrence_id"]]

    for i in range(1, len(visits)):
        next_start = visits.loc[i, "visit_start_datetime"]
        next_end = visits.loc[i, "visit_end_datetime"]
        next_visit_id = visits.loc[i, "visit_occurrence_id"]

        if (next_start - current_end).total_seconds() <= 48 * 60 * 60:
            current_end = max(current_end, next_end)
            current_visit_ids.append(next_visit_id)

        else:
            merged_rows.append({
                "start_datetime": current_start,
                "end_datetime": current_end,
                "visit_ids": current_visit_ids
            })

            current_start = next_start
            current_end = next_end
            current_visit_ids = [next_visit_id]

    merged_rows.append({
        "start_datetime": current_start,
        "end_datetime": current_end,
        "visit_ids": current_visit_ids
    })

    merged_hospitalizations = pd.DataFrame(merged_rows)

    # Enrich each hospitalization with visit, department, and patient information
    final_rows = []

    if department_keywords is None:
        department_keywords = []

    for idx, hosp in merged_hospitalizations.iterrows():
        hospitalization_number = idx + 1
        merged_hospitalization_id = f"{person_id}_{hospitalization_number}"

        visit_ids = hosp["visit_ids"]

        visit_types = []
        departments = []

        has_inpatient_visit = 0
        has_relevant_department = 0
        has_relevant_inpatient_visit = 0

        for visit_id in visit_ids:
            visit_type = str(visit_type_map.get(visit_id, ""))
            visit_types.append(visit_type)

            visit_departments = department_map.get(visit_id, [])
            departments.extend(visit_departments)

            is_inpatient = "inpatient" in visit_type.lower()

            is_relevant_department = any(
                keyword in department
                for department in visit_departments
                for keyword in department_keywords
            )

            if is_inpatient:
                has_inpatient_visit = 1

            if is_relevant_department:
                has_relevant_department = 1

            if is_inpatient and is_relevant_department:
                has_relevant_inpatient_visit = 1

        departments = sorted(set(departments))

        # Derive hospitalization duration and patient age at admission
        length_of_stay_days = (
            hosp["end_datetime"] - hosp["start_datetime"]
        ).total_seconds() / (3600 * 24)

        age_at_hospitalization = None

        if birth_year is not None and pd.notna(birth_year):
            age_at_hospitalization = (
                hosp["start_datetime"].year - int(birth_year)
            )

        final_rows.append({
            "person_id": str(person_id),
            "merged_hospitalization_id": merged_hospitalization_id,
            "hospitalization_number": hospitalization_number,
            "start_datetime": hosp["start_datetime"],
            "end_datetime": hosp["end_datetime"],
            "length_of_stay_days": length_of_stay_days,
            "age_at_hospitalization": age_at_hospitalization,
            "visit_ids": visit_ids,
            "visit_types": visit_types,
            "departments": departments,
            "has_inpatient_visit": has_inpatient_visit,
            "has_relevant_department": has_relevant_department,
            "has_relevant_inpatient_visit": has_relevant_inpatient_visit
        })

    return pd.DataFrame(final_rows, columns=output_columns)

In [ ]:
def build_all_merged_hospitalizations(
    visits_by_patient,
    birth_year_by_patient,
    visit_type_map,
    department_map,
    department_keywords
):
    """
    Build merged hospitalization episodes for all patients.

    Uses pre-built patient and visit lookup dictionaries and returns
    a single DataFrame containing all merged hospitalizations.
    """

    all_merged = []
    patient_ids = sorted(birth_year_by_patient.keys())

    # Build hospitalization episodes patient by patient
    for i, person_id in enumerate(patient_ids, start=1):
        patient_visits = visits_by_patient.get(person_id, pd.DataFrame())
        birth_year = birth_year_by_patient.get(person_id)

        merged_hospitalizations = build_merged_hospitalizations(
            patient_visits=patient_visits,
            person_id=person_id,
            birth_year=birth_year,
            visit_type_map=visit_type_map,
            department_map=department_map,
            department_keywords=department_keywords
        )

        if not merged_hospitalizations.empty:
            all_merged.append(merged_hospitalizations)

        # Report progress for large datasets
        if i % 1000 == 0:
            print(f"Processed {i}/{len(patient_ids)} patients")

    if not all_merged:
        return pd.DataFrame()

    return pd.concat(all_merged, ignore_index=True)S

## 3. Patient Features


In [ ]:
# Clinical concept keywords used for feature extraction

abscess_drainage_terms = [
    "perianal abscess",
    "ischiorectal",
    "intramural abscess",
    "appendix abscess",
    "abdominal abscess",
    "abdominal wall abscess",
    "subdiaphragmatic abscess",
    "subphrenic abscess",
    "perivesical"
]

bowel_resection_terms = [
    "colectomy",
    "hemicolectomy",
    "ileocolectomy",
    "ileocolic",
    "small bowel",
    "small intestine",
    "enterectomy",
    "resection of ileum",
    "bowel resection",
    "abdominoperineal resection"
]

corticosteroid_terms = [
    "prednisone",
    "prednisolone",
    "methylprednisolone",
    "hydrocortisone",
    "budesonide"
]

biologic_terms = [
    "infliximab",
    "adalimumab",
    "ustekinumab"
]

bowel_obstruction_terms = [
    "intestinal obstruction"
]

body_temperature_terms = [
    "body temperature",
    "oral temperature",
    "rectal temperature",
    "axillary temperature"
]

heart_rate_terms = [
    "heart rate",
    "pulse rate"
]

bmi_terms = [
    "body mass index",
    "bmi"
]

body_weight_terms = [
    "body weight",
    "weight"
]

leukocyte_terms = [
    "leukocyte",
    "wbc",
    "white blood cell"
]

platelet_terms = [
    "platelet",
    "platelet count",
    "thrombocyte"
]

hemoglobin_terms = [
    "hemoglobin",
    "haemoglobin",
    "hb"
]

albumin_terms = [
    "albumin",
    "serum albumin"
]

crp_terms = [
    "c reactive protein",
    "crp",
    "c-reactive protein"
]

# Institution-specific department terms are configured locally
relevant_department_terms = []

In [ ]:
def count_total_visits(patient_visits):
    """Count unique visits for a single patient."""

    return patient_visits["visit_occurrence_id"].nunique()

In [ ]:
def get_demographic_features(patient):
    """Extract basic demographic features for a single patient."""

    year_of_birth = int(patient["year_of_birth"].iloc[0])
    gender_concept_id = patient["gender_concept_name"].iloc[0]

    current_year = pd.Timestamp.today().year
    # age = current_year - year_of_birth

    demographic_features = {
        "birth_year": year_of_birth,
        "gender": gender_concept_id
    }

    return demographic_features

In [ ]:
def has_smoking(patient_conditions, patient_observations):
    """Check for smoking-related records in condition or observation data."""

    condition_match = (
        patient_conditions["condition_concept_name"]
        .fillna("")
        .str.lower()
        .str.contains("smoking|tobacco|nicotine", regex=True)
        .any()
    )

    observation_match = (
        patient_observations["observation_concept_name"]
        .fillna("")
        .str.lower()
        .str.contains("smoking|tobacco|nicotine", regex=True)
        .any()
    )

    return int(condition_match or observation_match)

In [ ]:
def is_alive(person_id, death_df):
    """Return 1 if the patient is alive and 0 if deceased."""

    return 0 if str(person_id) in set(death_df["person_id"].astype(str)) else 1

In [ ]:
def count_merged_hospitalizations(merged_hospitalizations):
    """Count merged hospitalization episodes for a single patient."""

    return len(merged_hospitalizations)

In [ ]:
def get_observation_codes(patient_observations, keywords):
    """Return matching observation concepts and source values."""

    matches = patient_observations[
        patient_observations["observation_concept_name"]
        .fillna("")
        .str.lower()
        .str.contains("|".join(keywords), regex=True)
    ].copy()

    if matches.empty:
        return None

    matches = matches.drop_duplicates(
        subset=[
            "observation_concept_id",
            "observation_concept_name",
            "observation_source_value"
        ]
    )

    return " | ".join(
        matches.apply(
            lambda row: (
                f"{row['observation_concept_name']} "
                f"({row['observation_concept_id']}; "
                f"source={row['observation_source_value']})"
            ),
            axis=1
        ).astype(str).unique()
    )

In [ ]:
def get_first_diagnosis_hospitalization(
    patient_conditions,
    merged_hospitalizations,
    diagnosis_name
):
    """Return the first merged hospitalization containing a diagnosis."""

    diagnosis_conditions = patient_conditions[
        patient_conditions["condition_concept_name"]
        .fillna("")
        .str.lower()
        .str.contains(diagnosis_name, regex=False)
    ].copy()

    if diagnosis_conditions.empty or merged_hospitalizations.empty:
        return None

    diagnosis_visit_ids = set(
        diagnosis_conditions["visit_occurrence_id"]
        .dropna()
        .unique()
    )

    for _, hospitalization in merged_hospitalizations.iterrows():
        hospitalization_visit_ids = set(
            hospitalization["visit_ids"]
        )

        if hospitalization_visit_ids & diagnosis_visit_ids:
            hosp_num = hospitalization["hospitalization_number"]

            return f"{hosp_num}"

    return None

In [ ]:
def is_crohn(patient_conditions, patient_observations):
    """
    Return 1 if Crohn's disease appears in either the condition
    or observation table, otherwise return 0.
    """

    condition_crohn = (
        patient_conditions["condition_concept_name"]
        .fillna("")
        .str.lower()
        .str.contains("crohn", regex=False)
        .any()
    )

    observation_crohn = (
        patient_observations["observation_concept_name"]
        .fillna("")
        .str.lower()
        .str.contains("crohn", regex=False)
        .any()
    )

    return int(condition_crohn or observation_crohn)

In [ ]:
def get_age_at_first_diagnosis(
    patient,
    patient_conditions,
    diagnosis_keywords
):
    """Calculate age at the first recorded matching diagnosis."""

    diagnosis_conditions = patient_conditions[
        patient_conditions["condition_concept_name"]
        .fillna("")
        .str.lower()
        .str.contains("|".join(diagnosis_keywords), regex=True)
    ].copy()

    if diagnosis_conditions.empty:
        return None

    diagnosis_conditions["condition_start_date"] = pd.to_datetime(
        diagnosis_conditions["condition_start_date"],
        errors="coerce"
    )

    first_diagnosis_date = diagnosis_conditions["condition_start_date"].min()

    if pd.isna(first_diagnosis_date):
        return None

    birth_year = int(patient["year_of_birth"].iloc[0])

    return first_diagnosis_date.year - birth_year

In [ ]:
def count_hospitalizations_in_age_range(
    merged_hospitalizations,
    min_age,
    max_age=None
):
    """Count hospitalizations within the requested age range."""

    if merged_hospitalizations.empty:
        return 0

    ages = pd.to_numeric(
        merged_hospitalizations["age_at_hospitalization"],
        errors="coerce"
    )

    if max_age is None:
        return (ages >= min_age).sum()

    return ((ages >= min_age) & (ages <= max_age)).sum()

In [ ]:
def has_condition(patient_conditions, keywords):
    """Check whether any condition matches the given keywords."""

    condition_names = (
        patient_conditions["condition_concept_name"]
        .fillna("")
        .str.lower()
    )

    keywords = [keyword.lower() for keyword in keywords]

    for keyword in keywords:
        if condition_names.str.contains(keyword, regex=False).any():
            return 1

    return 0

In [ ]:
def has_procedure(patient_procedures, keywords):
    """Check whether any procedure matches the given keywords."""

    procedure_names = (
        patient_procedures["procedure_concept_name"]
        .fillna("")
        .str.lower()
    )

    keywords = [keyword.lower() for keyword in keywords]

    for keyword in keywords:
        if procedure_names.str.contains(keyword.lower(), regex=False).any():
            return 1

    return 0

In [ ]:
def has_drug(patient_drugs, keywords):
    """Check whether any drug matches the given keywords."""

    drug_names = (
        patient_drugs["drug_concept_name"]
        .fillna("")
        .str.lower()
    )

    keywords = [keyword.lower() for keyword in keywords]

    for keyword in keywords:
        if drug_names.str.contains(keyword, regex=False).any():
            return 1

    return 0

In [ ]:
def get_diagnosis_hospitalization_list(
    patient_conditions,
    merged_hospitalizations,
    diagnosis_keywords
):
    """Return merged hospitalizations containing selected diagnoses."""

    diagnosis_conditions = patient_conditions[
        patient_conditions["condition_concept_name"]
        .fillna("")
        .str.lower()
        .str.contains("|".join(diagnosis_keywords), regex=True)
    ].copy()

    if diagnosis_conditions.empty or merged_hospitalizations.empty:
        return None

    diagnosis_conditions["visit_occurrence_id"] = (
        diagnosis_conditions["visit_occurrence_id"].astype(str)
    )

    hospitalization_parts = []

    for _, hospitalization in merged_hospitalizations.iterrows():
        hospitalization_number = hospitalization["hospitalization_number"]
        hospitalization_visit_ids = [
            str(visit_id) for visit_id in hospitalization["visit_ids"]
        ]

        diagnoses_in_hospitalization = diagnosis_conditions[
            diagnosis_conditions["visit_occurrence_id"].isin(
                hospitalization_visit_ids
            )
        ]

        if diagnoses_in_hospitalization.empty:
            continue

        diagnosis_names = "; ".join(
            diagnoses_in_hospitalization["condition_concept_name"]
            .dropna()
            .astype(str)
            .unique()
        )

        hospitalization_parts.append(
            f"Hosp{hospitalization_number}: {diagnosis_names}"
        )

    if not hospitalization_parts:
        return None

    return " | ".join(hospitalization_parts)

In [ ]:
def count_hospitalizations_with_condition(
    patient_conditions,
    merged_hospitalizations,
    keywords
):
    """Count merged hospitalizations containing a selected condition."""

    matching_conditions = patient_conditions[
        patient_conditions["condition_concept_name"]
        .fillna("")
        .str.lower()
        .str.contains("|".join(keywords), regex=True)
    ]

    if matching_conditions.empty:
        return 0

    condition_visit_ids = set(
        matching_conditions["visit_occurrence_id"].dropna()
    )

    hospitalization_count = 0

    for _, hospitalization in merged_hospitalizations.iterrows():
        hospitalization_visit_ids = set(
            hospitalization["visit_ids"]
        )

        if hospitalization_visit_ids & condition_visit_ids:
            hospitalization_count += 1

    return hospitalization_count

In [ ]:
def get_measurement_stats_per_hospitalization(
    patient_measurements,
    merged_hospitalizations,
    measurement_keywords
):
    """
    Return the first measurement recorded within the first 72 hours
    of each merged hospitalization.
    """

    if patient_measurements.empty or merged_hospitalizations.empty:
        return None

    measurements = patient_measurements.copy()

    measurement_names = (
        measurements["measurement_concept_name"]
        .fillna("")
        .str.lower()
    )

    selected_measurements = measurements[
        measurement_names.str.contains(
            "|".join(measurement_keywords),
            regex=True
        )
    ].copy()

    if selected_measurements.empty:
        return None

    selected_measurements["value_as_number"] = pd.to_numeric(
        selected_measurements["value_as_number"],
        errors="coerce"
    )

    selected_measurements = selected_measurements.dropna(
        subset=[
            "value_as_number",
            "visit_occurrence_id",
            "measurement_datetime"
        ]
    )

    selected_measurements = selected_measurements.drop_duplicates(
        subset=[
            "person_id",
            "visit_occurrence_id",
            "measurement_concept_name",
            "measurement_datetime",
            "value_as_number"
        ]
    )

    hospitalization_stats = []

    # Extract the first available measurement within 72 hours of admission
    for _, hospitalization in merged_hospitalizations.iterrows():
        hospitalization_number = hospitalization["hospitalization_number"]
        hospitalization_start = hospitalization["start_datetime"]
        hospitalization_visit_ids = set(hospitalization["visit_ids"])

        measurements_in_hosp = selected_measurements[
            selected_measurements["visit_occurrence_id"].isin(
                hospitalization_visit_ids
            )
        ].copy()

        if measurements_in_hosp.empty:
            continue

        window_72h_end = hospitalization_start + pd.Timedelta(hours=72)

        measurements_72h = measurements_in_hosp[
            (measurements_in_hosp["measurement_datetime"] >= hospitalization_start) &
            (measurements_in_hosp["measurement_datetime"] <= window_72h_end)
        ].copy()

        if measurements_72h.empty:
            continue

        measurements_72h = measurements_72h.sort_values(
            "measurement_datetime"
        )

        first_value = measurements_72h.iloc[0]["value_as_number"]

        hospitalization_stats.append(
            f"Hosp{hospitalization_number}: {first_value}"
        )

    if not hospitalization_stats:
        return None

    return " | ".join(hospitalization_stats)

In [ ]:
def get_days_since_previous_hospitalization(
    merged_hospitalizations,
    first_crohn_hospitalization
):
    """
    Calculate the number of days between the previous hospitalization
    and the first Crohn's hospitalization.
    """

    if merged_hospitalizations.empty or first_crohn_hospitalization is None:
        return np.nan

    hospitalizations = merged_hospitalizations.copy()

    hospitalizations["hospitalization_number"] = pd.to_numeric(
        hospitalizations["hospitalization_number"],
        errors="coerce"
    )

    hospitalizations["start_datetime"] = pd.to_datetime(
        hospitalizations["start_datetime"],
        errors="coerce"
    )

    hospitalizations["end_datetime"] = pd.to_datetime(
        hospitalizations["end_datetime"],
        errors="coerce"
    )

    first_crohn_hospitalization = pd.to_numeric(
        first_crohn_hospitalization,
        errors="coerce"
    )

    index_hospitalization = hospitalizations[
        hospitalizations["hospitalization_number"]
        == first_crohn_hospitalization
    ]

    if index_hospitalization.empty:
        return np.nan

    index_start = index_hospitalization["start_datetime"].min()

    if pd.isna(index_start):
        return np.nan

    previous_hospitalizations = hospitalizations[
        hospitalizations["end_datetime"] < index_start
    ]

    if previous_hospitalizations.empty:
        return np.nan

    previous_end = previous_hospitalizations["end_datetime"].max()

    return (index_start - previous_end).days

In [ ]:
def count_hospitalizations_last_year_before_crohn(
    merged_hospitalizations,
    first_crohn_hospitalization
):
    """
    Count merged hospitalizations during the 365 days
    before the first Crohn's hospitalization.
    """

    if merged_hospitalizations.empty or first_crohn_hospitalization is None:
        return 0

    hospitalizations = merged_hospitalizations.copy()

    hospitalizations["hospitalization_number"] = pd.to_numeric(
        hospitalizations["hospitalization_number"],
        errors="coerce"
    )

    hospitalizations["start_datetime"] = pd.to_datetime(
        hospitalizations["start_datetime"],
        errors="coerce"
    )

    first_crohn_hospitalization = pd.to_numeric(
        first_crohn_hospitalization,
        errors="coerce"
    )

    index_hospitalization = hospitalizations[
        hospitalizations["hospitalization_number"]
        == first_crohn_hospitalization
    ]

    if index_hospitalization.empty:
        return 0

    index_start = index_hospitalization["start_datetime"].min()

    if pd.isna(index_start):
        return 0

    one_year_before = index_start - pd.Timedelta(days=365)

    previous_year_hospitalizations = hospitalizations[
        (hospitalizations["start_datetime"] >= one_year_before)
        & (hospitalizations["start_datetime"] < index_start)
    ]

    return previous_year_hospitalizations[
        "hospitalization_number"
    ].nunique()

In [ ]:
def build_patient_features(
    person_id,
    persons_by_patient,
    visits_by_patient,
    conditions_by_patient,
    observations_by_patient,
    procedures_by_patient,
    drugs_by_patient,
    measurements_by_patient,
    merged_hospitalizations_by_patient,
    empty_merged_hospitalizations
):
    """Build a patient-level feature row for a single patient."""

    person_id = str(person_id)

    patient = persons_by_patient.get(person_id, pd.DataFrame())

    if patient.empty:
        print(f"No patient found for person_id={person_id}")
        return pd.DataFrame()

    # Retrieve patient-specific OMOP records
    patient_visits = visits_by_patient.get(person_id, empty_visits.copy())
    patient_conditions = conditions_by_patient.get(person_id, empty_conditions.copy())
    patient_observations = observations_by_patient.get(person_id, empty_observations.copy())
    patient_procedures = procedures_by_patient.get(person_id, empty_procedures.copy())
    patient_measurements = measurements_by_patient.get(person_id, empty_measurements.copy())
    patient_drugs = drugs_by_patient.get(person_id, empty_drugs.copy())

    # Retrieve hospitalization history
    merged_hospitalizations = merged_hospitalizations_by_patient.get(
        person_id,
        empty_merged_hospitalizations.copy()
    )

    merged_hospitalizations_in_departments = merged_hospitalizations[
        merged_hospitalizations["has_relevant_inpatient_visit"] == 1
    ].copy()

    demographic_features = get_demographic_features(patient)

    first_crohn_hospitalization = get_first_diagnosis_hospitalization(
        patient_conditions,
        merged_hospitalizations,
        "crohn"
    )

    # Assemble patient-level features
    return pd.DataFrame([{
        "patient_id": person_id,
        "birth_year": demographic_features["birth_year"],
        "is_alive": is_alive(person_id, death_df),
        "sex": demographic_features["gender"],
        "smoking": has_smoking(patient_conditions, patient_observations),

        "number_of_visits": count_total_visits(patient_visits),
        "number_of_hospitalizations": count_merged_hospitalizations(
            merged_hospitalizations
        ),
        "number_of_relevant_hospitalizations": count_merged_hospitalizations(
            merged_hospitalizations_in_departments
        ),

        "hospitalizations_age_18_35": count_hospitalizations_in_age_range(
            merged_hospitalizations_in_departments, 18, 35
        ),
        "hospitalizations_age_36_55": count_hospitalizations_in_age_range(
            merged_hospitalizations_in_departments, 36, 55
        ),
        "hospitalizations_age_56_plus": count_hospitalizations_in_age_range(
            merged_hospitalizations_in_departments, 56, None
        ),

        "first_crohn_hospitalization": first_crohn_hospitalization,
        "days_since_previous_hospitalization": get_days_since_previous_hospitalization(
            merged_hospitalizations,
            first_crohn_hospitalization
        ),
        "hospitalizations_last_year_before_crohn":
            count_hospitalizations_last_year_before_crohn(
                merged_hospitalizations,
                first_crohn_hospitalization
            ),

        "is_crohn": is_crohn(patient_conditions, patient_observations),
        "crohn_observation": get_observation_codes(
            patient_observations, ["crohn"]
        ),
        "age_at_first_crohn_diagnosis": get_age_at_first_diagnosis(
            patient, patient_conditions, ["crohn"]
        ),
        "crohn_diagnosis_list": get_diagnosis_hospitalization_list(
            patient_conditions,
            merged_hospitalizations,
            ["crohn"]
        ),

        "colitis_observation": get_observation_codes(
            patient_observations, ["colitis"]
        ),
        "colitis_diagnosis_list": get_diagnosis_hospitalization_list(
            patient_conditions,
            merged_hospitalizations,
            ["colitis"]
        ),

        "has_fistula": has_condition(patient_conditions, ["fistula"]),
        "has_abscess": has_condition(patient_conditions, ["abscess"]),
        "has_fissure": has_condition(patient_conditions, ["fissure"]),
        "has_bowel_obstruction": has_condition(
            patient_conditions, bowel_obstruction_terms
        ),

        "has_bowel_resection": has_procedure(
            patient_procedures, bowel_resection_terms
        ),
        "has_abscess_drainage": has_procedure(
            patient_procedures, abscess_drainage_terms
        ),

        "has_corticosteroids": has_drug(
            patient_drugs, corticosteroid_terms
        ),
        "has_biological_treatment": has_drug(
            patient_drugs, biologic_terms
        ),

        "temperature": get_measurement_stats_per_hospitalization(
            patient_measurements,
            merged_hospitalizations,
            body_temperature_terms
        ),
        "heart_rate": get_measurement_stats_per_hospitalization(
            patient_measurements,
            merged_hospitalizations,
            heart_rate_terms
        ),
        "bmi": get_measurement_stats_per_hospitalization(
            patient_measurements,
            merged_hospitalizations,
            bmi_terms
        ),
        "weight": get_measurement_stats_per_hospitalization(
            patient_measurements,
            merged_hospitalizations,
            body_weight_terms
        ),

        "diarrhea_hospitalization_count": count_hospitalizations_with_condition(
            patient_conditions,
            merged_hospitalizations,
            ["diarrhea"]
        ),
        "abdominal_pain_hospitalization_count":
            count_hospitalizations_with_condition(
                patient_conditions,
                merged_hospitalizations,
                ["abdominal pain"]
            ),

        "leukocytosis": get_measurement_stats_per_hospitalization(
            patient_measurements,
            merged_hospitalizations,
            leukocyte_terms
        ),
        "thrombocytosis": get_measurement_stats_per_hospitalization(
            patient_measurements,
            merged_hospitalizations,
            platelet_terms
        ),
        "hemoglobin": get_measurement_stats_per_hospitalization(
            patient_measurements,
            merged_hospitalizations,
            hemoglobin_terms
        ),
        "albumin": get_measurement_stats_per_hospitalization(
            patient_measurements,
            merged_hospitalizations,
            albumin_terms
        ),
        "crp": get_measurement_stats_per_hospitalization(
            patient_measurements,
            merged_hospitalizations,
            crp_terms
        )
    }])

## 4. Severity Score



In [ ]:
def feature_score(row, feature_name, weight, condition_func=None):
    """
    Calculate the score contribution of a single feature.

    If a condition function is provided, the score is added only
    when the condition is satisfied. Missing values contribute 0.
    """

    value = row.get(feature_name, 0)

    if pd.isna(value):
        return 0

    if condition_func is not None and not condition_func(value):
        return 0

    return value * weight

In [ ]:
def hospitalization_count_score(count, weight=25, max_count=20):
    """
    Calculate a score from a hospitalization count.

    The contribution is capped at max_count hospitalizations.
    """

    if pd.isna(count):
        return 0

    return weight * min(count, max_count)

In [ ]:
def extract_hospitalization_values(value):
    """
    Extract numeric measurement values from hospitalization strings.

    Example
    -------
    Input:
        "Hosp24: 36.4 | Hosp25: 38.1 | Hosp26: 37.0"

    Output:
        [36.4, 38.1, 37.0]
    """

    if value is None or pd.isna(value):
        return []

    numbers = re.findall(
        r"Hosp\d+:\s*(-?\d+\.?\d*)",
        str(value)
    )

    return [float(num) for num in numbers]

In [ ]:
def measurement_threshold_score(
    row,
    feature_name,
    threshold_func,
    weight=25,
    max_count=20
):
    """
    Calculate a severity score for a measurement feature.

    Each hospitalization contributes when its measurement value
    crosses the specified clinical threshold.
    """

    values = extract_hospitalization_values(
        row.get(feature_name)
    )

    abnormal_count = sum(
        1
        for value in values
        if threshold_func(value, row)
    )

    return weight * min(abnormal_count, max_count)

In [ ]:
def calculate_severe_status_counter(row, averages=None):
    """Calculate the total severity score for one patient."""

    score = 0

    score += feature_score(row, "smoking", 900)

    if averages is not None:
        if row["hospitalizations_age_18_35"] > averages["hospitalizations_age_18_35"]:
            score += 1000
        if row["hospitalizations_age_36_55"] > averages["hospitalizations_age_36_55"]:
            score += 1000
        if row["hospitalizations_age_56_plus"] > averages["hospitalizations_age_56_plus"]:
            score += 1000

    score += feature_score(row, "is_crohn", 100000)
    score += feature_score(row, "has_fistula", 1000)

    score += feature_score(
        row,
        "has_abscess",
        1000,
        condition_func=lambda v: row["is_crohn"] == 1
    )

    score += feature_score(
        row,
        "has_fissure",
        1000,
        condition_func=lambda v: row["is_crohn"] == 1
    )

    score += feature_score(row, "has_bowel_obstruction", 1000)
    score += feature_score(row, "has_bowel_resection", 1000)
    score += feature_score(row, "has_abscess_drainage", 1000)
    score += feature_score(row, "has_corticosteroids", 1000)
    score += feature_score(row, "has_biological_treatment", 1000)

    score += measurement_threshold_score(
        row, "temperature", lambda v, row: v > 37.8
    )
    score += measurement_threshold_score(
        row, "heart_rate", lambda v, row: v > 100
    )
    score += measurement_threshold_score(
        row,
        "bmi",
        lambda v, row: v < 20
        if (pd.Timestamp.today().year - row["birth_year"]) < 70
        else v < 22,
        50
    )

    score += hospitalization_count_score(
        row["diarrhea_hospitalization_count"]
    )
    score += hospitalization_count_score(
        row["abdominal_pain_hospitalization_count"]
    )

    score += measurement_threshold_score(
        row, "leukocytosis", lambda v, row: v > 11
    )
    score += measurement_threshold_score(
        row, "thrombocytosis", lambda v, row: v > 450
    )
    score += measurement_threshold_score(
        row,
        "hemoglobin",
        lambda v, row: v < 11.7 if row["sex"] == "FEMALE" else v < 13.5
    )
    score += measurement_threshold_score(
        row, "albumin", lambda v, row: v < 34
    )
    score += measurement_threshold_score(
        row, "crp", lambda v, row: v > 5
    )

    return score

In [ ]:
def calculate_severity_scores(row, averages=None):
    """Calculate individual severity-score contributions for one patient."""

    scores = {}

    scores["score_smoking"] = feature_score(row, "smoking", 900)

    if averages is not None:
        scores["score_hospitalizations_age_18_35"] = (
            1000
            if row["hospitalizations_age_18_35"]
            > averages["hospitalizations_age_18_35"]
            else 0
        )
        scores["score_hospitalizations_age_36_55"] = (
            1000
            if row["hospitalizations_age_36_55"]
            > averages["hospitalizations_age_36_55"]
            else 0
        )
        scores["score_hospitalizations_age_56_plus"] = (
            1000
            if row["hospitalizations_age_56_plus"]
            > averages["hospitalizations_age_56_plus"]
            else 0
        )
    else:
        scores["score_hospitalizations_age_18_35"] = 0
        scores["score_hospitalizations_age_36_55"] = 0
        scores["score_hospitalizations_age_56_plus"] = 0

    scores["score_is_crohn"] = feature_score(row, "is_crohn", 100000)
    scores["score_has_fistula"] = feature_score(row, "has_fistula", 1000)

    scores["score_has_abscess"] = feature_score(
        row,
        "has_abscess",
        1000,
        condition_func=lambda v: row["is_crohn"] == 1
    )
    scores["score_has_fissure"] = feature_score(
        row,
        "has_fissure",
        1000,
        condition_func=lambda v: row["is_crohn"] == 1
    )

    scores["score_has_bowel_obstruction"] = feature_score(
        row, "has_bowel_obstruction", 1000
    )
    scores["score_has_bowel_resection"] = feature_score(
        row, "has_bowel_resection", 1000
    )
    scores["score_has_abscess_drainage"] = feature_score(
        row, "has_abscess_drainage", 1000
    )
    scores["score_has_corticosteroids"] = feature_score(
        row, "has_corticosteroids", 1000
    )
    scores["score_has_biological_treatment"] = feature_score(
        row, "has_biological_treatment", 1000
    )

    scores["score_temperature"] = measurement_threshold_score(
        row, "temperature", lambda v, row: v > 37.8
    )
    scores["score_heart_rate"] = measurement_threshold_score(
        row, "heart_rate", lambda v, row: v > 100
    )
    scores["score_bmi"] = measurement_threshold_score(
        row,
        "bmi",
        lambda v, row: v < 20
        if (pd.Timestamp.today().year - row["birth_year"]) < 70
        else v < 22,
        50
    )

    scores["score_diarrhea"] = hospitalization_count_score(
        row["diarrhea_hospitalization_count"]
    )
    scores["score_abdominal_pain"] = hospitalization_count_score(
        row["abdominal_pain_hospitalization_count"]
    )

    scores["score_leukocytosis"] = measurement_threshold_score(
        row, "leukocytosis", lambda v, row: v > 11
    )
    scores["score_thrombocytosis"] = measurement_threshold_score(
        row, "thrombocytosis", lambda v, row: v > 450
    )
    scores["score_hemoglobin"] = measurement_threshold_score(
        row,
        "hemoglobin",
        lambda v, row: v < 11.7 if row["sex"] == "FEMALE" else v < 13.5
    )
    scores["score_albumin"] = measurement_threshold_score(
        row, "albumin", lambda v, row: v < 34
    )
    scores["score_crp"] = measurement_threshold_score(
        row, "crp", lambda v, row: v > 5
    )

    return scores

## 5. Build Feature Dataset

In [ ]:
import ast

# Cache merged hospitalizations to avoid rebuilding them on every run
cache_folder = "cache"
os.makedirs(cache_folder, exist_ok=True)

merged_file = os.path.join(
    cache_folder,
    f"{hospital_name}_merged_hospitalizations.csv"
)

if os.path.exists(merged_file):

    print(f"Loading merged hospitalizations for {hospital_name}...")

    merged_hospitalizations_all = pd.read_csv(
        merged_file,
        converters={
            "visit_ids": ast.literal_eval,
            "visit_types": ast.literal_eval,
            "departments": ast.literal_eval
        }
    )

    # Restore data types after loading from CSV
    merged_hospitalizations_all["person_id"] = (
        merged_hospitalizations_all["person_id"].astype(str)
    )

    merged_hospitalizations_all["start_datetime"] = pd.to_datetime(
        merged_hospitalizations_all["start_datetime"],
        errors="coerce"
    )

    merged_hospitalizations_all["end_datetime"] = pd.to_datetime(
        merged_hospitalizations_all["end_datetime"],
        errors="coerce"
    )

else:

    print(f"Building merged hospitalizations for {hospital_name}...")

    start_time = time.time()

    merged_hospitalizations_all = build_all_merged_hospitalizations(
        visits_by_patient,
        birth_year_by_patient,
        visit_type_map,
        department_map,
        relevant_department_terms
    )

    end_time = time.time()

    print(f"Time: {(end_time - start_time) / 60:.2f} minutes")
    print("Finished building merged hospitalizations for all patients successfully")

    merged_hospitalizations_all.to_csv(
        merged_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved merged hospitalizations to {merged_file}")

In [ ]:
# Build patient feature dataset

output_folder = "features"
os.makedirs(output_folder, exist_ok=True)

start_time = time.time()

# Department definitions are institution-specific and should be configured locally
if relevant_department_terms is None:
    raise ValueError(
        "Configure relevant_department_terms before running the pipeline."
    )

# Ensure patient IDs use a consistent type
merged_hospitalizations_all["person_id"] = (
    merged_hospitalizations_all["person_id"].astype(str)
)

# Group merged hospitalizations by patient
merged_hospitalizations_by_patient = dict(
    tuple(
        merged_hospitalizations_all.groupby("person_id")
    )
)

empty_merged_hospitalizations = pd.DataFrame(
    columns=merged_hospitalizations_all.columns
)

# Build features for all available patients
person_ids = sorted(persons_by_patient.keys())

print(f"Patients in {hospital_name}: {len(person_ids):,}")

all_features = []

for i, person_id in enumerate(person_ids, start=1):
    try:
        patient_features = build_patient_features(
            person_id,
            persons_by_patient,
            visits_by_patient,
            conditions_by_patient,
            observations_by_patient,
            procedures_by_patient,
            drugs_by_patient,
            measurements_by_patient,
            merged_hospitalizations_by_patient,
            empty_merged_hospitalizations
        )

        if patient_features is not None and not patient_features.empty:
            all_features.append(patient_features)

    except Exception as e:
        print(
            f"Error for patient {person_id}: "
            f"{type(e).__name__}: {e}"
        )
        raise

    if i % 500 == 0:
        print(f"Processed {i}/{len(person_ids)} patients")

if not all_features:
    raise ValueError("No patient features were generated.")

patient_features_df = pd.concat(
    all_features,
    ignore_index=True
)

# Keep patients who were adults at their first recorded Crohn's diagnosis
patient_features_df = patient_features_df[
    patient_features_df["age_at_first_crohn_diagnosis"].notna()
    & (patient_features_df["age_at_first_crohn_diagnosis"] >= 18)
].reset_index(drop=True)

print(
    f"Adult Crohn's patients: "
    f"{patient_features_df['patient_id'].nunique():,}"
)

# Calculate cohort averages used by the current severity score
averages = {
    "hospitalizations_age_18_35":
        patient_features_df["hospitalizations_age_18_35"].mean(),
    "hospitalizations_age_36_55":
        patient_features_df["hospitalizations_age_36_55"].mean(),
    "hospitalizations_age_56_plus":
        patient_features_df["hospitalizations_age_56_plus"].mean()
}

# Calculate total severity score
patient_features_df["severe_status_counter"] = (
    patient_features_df.apply(
        lambda row: calculate_severe_status_counter(
            row,
            averages
        ),
        axis=1
    )
)

# Calculate score contribution of each feature
score_df = patient_features_df.apply(
    lambda row: pd.Series(
        calculate_severity_scores(
            row,
            averages
        )
    ),
    axis=1
)

patient_features_df = pd.concat(
    [patient_features_df, score_df],
    axis=1
)

# Sort patients by severity score
patient_features_df = patient_features_df.sort_values(
    by="severe_status_counter",
    ascending=False
).reset_index(drop=True)

# Move the total severity score next to patient_id
cols = patient_features_df.columns.tolist()
cols.remove("severe_status_counter")
cols.insert(1, "severe_status_counter")
patient_features_df = patient_features_df[cols]

# Export feature dataset
output_path = os.path.join(
    output_folder,
    f"{hospital_name}_patient_features.csv"
)

patient_features_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

end_time = time.time()

print(f"File saved: {output_path}")
print(f"Time: {(end_time - start_time) / 60:.2f} minutes")
print("Finished successfully")